In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [4]:
# ================================================================
# Imports
# ================================================================
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix, bmat, diags
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import geopandas as gpd
import pyreadr

# ================================================================
# Load data (remove isolated points)
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0]

all_y = snow.drop(index=no_nbs).reset_index(drop=True)

coords = all_y.iloc[:, :2].to_numpy()
y      = all_y.iloc[:, 2:].to_numpy()

S, TT = y.shape
period = 52

print(f"Spatial locations used (non-isolated): {S}")

# ================================================================
# Global time trend (scale ONCE)
# ================================================================
t_full = np.arange(1, TT + 1)
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)

# ================================================================
# Load temperature & scale globally
# ================================================================
snow_temp = pyreadr.read_r("snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)

temp = snow_temp.drop(index=no_nbs).iloc[:, 2:].to_numpy()

temp_mean = temp.mean()
temp_sd   = temp.std(ddof=0)
temp_scaled_full = (temp - temp_mean) / temp_sd

# ================================================================
# Latitude & elevation (global scale)
# ================================================================
lat_raw = coords[:, 1]
lat = (lat_raw - lat_raw.mean()) / lat_raw.std(ddof=1)

elev_raw = pd.read_csv("curr_elev.csv").iloc[:, 3].to_numpy()
elev = (elev_raw - elev_raw.mean()) / elev_raw.std(ddof=1)

# ================================================================
# Build adjacency
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:, 0], coords[:, 1]),
    crs="EPSG:4326"
)

gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T

dif = xy[1] - xy[2]
theta = np.arctan2(dif[1], dif[0])
R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

rotated = (xy @ R.T) / 1e6
Distances = squareform(pdist(rotated))

Omg = (Distances <= 0.22).astype(int)
np.fill_diagonal(Omg, 0)
Omg = csr_matrix(Omg)

deg = np.array(Omg.sum(axis=1)).flatten()
assert np.all(deg > 0)

D = diags(deg)
prec = D - Omg   # ICAR

# ================================================================
# MCMC settings
# ================================================================
burn = 1000
thin = 5
tot_save = 1000

a_tau = 2.0
b_tau = 25.0

# ================================================================
# BYM++factor runner
# ================================================================
def run_bym_factor(event_name, loc_mask, kappa_transform, save_path):

    loc = np.where(loc_mask)
    pairs = np.column_stack(loc)
    pairs = pairs[np.lexsort((pairs[:, 0], pairs[:, 1]))]
    pairs[:, 1] += 1

    row_idx  = pairs[:, 0]
    time_idx = pairs[:, 1] - 1
    N = len(row_idx)

    next_y = y[pairs[:, 0], pairs[:, 1]]
    kappa  = kappa_transform(next_y)

    # ------------------------------------------------------------
    # Time variables
    # ------------------------------------------------------------
    t_raw   = time_idx + 1                  # for sin/cos
    t_trend = t_trend_full[time_idx]        # subset ONLY

    # ------------------------------------------------------------
    # Covariates (BYM part)
    # ------------------------------------------------------------
    covariates = np.column_stack([
        np.ones(N), np.ones(N),
        np.cos(2*np.pi*t_raw / period),
        np.cos(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        t_trend, t_trend
    ])

    K = covariates.shape[1]   # 8
    theta_dim = K * S + 3     # +3 factor parameters

    # ------------------------------------------------------------
    # BYM design matrix
    # ------------------------------------------------------------
    rows, cols, vals = [], [], []
    for i in tqdm(range(N), desc=f"Design {event_name}"):
        s = row_idx[i]
        for k in range(K):
            rows.append(i)
            cols.append(s + k*S)
            vals.append(covariates[i, k])

    X_bym = coo_matrix((vals, (rows, cols)), shape=(N, K*S)).tocsr()

    # ------------------------------------------------------------
    # Factor design (trend × spatial covariates)
    # ------------------------------------------------------------
    t_lat  = t_trend * lat[row_idx]
    t_elev = t_trend * elev[row_idx]
    t_temp = t_trend * temp_scaled_full[row_idx, time_idx]

    X_fac = csr_matrix(np.column_stack([t_lat, t_elev, t_temp]))

    from scipy.sparse import hstack

    X = hstack([X_bym, X_fac], format="csr")

    # ------------------------------------------------------------
    # Storage
    # ------------------------------------------------------------
    total_iters = burn + tot_save * thin
    all_theta = np.zeros((theta_dim, tot_save))
    all_tau   = np.zeros((K, tot_save))

    curr_theta = np.zeros(theta_dim)
    curr_tau   = np.ones(K)

    save_idx = 0

    # ------------------------------------------------------------
    # MCMC
    # ------------------------------------------------------------
    for it in tqdm(range(total_iters), desc=f"MCMC {event_name}"):

        phi = X @ curr_theta
        omega = random_polyagamma(1, phi, size=N)

        # ---- prior precision
        block_list = []
        for j in range(K):
            if j % 2 == 0:
                block_list.append((1 / curr_tau[j]) * prec)
            else:
                block_list.append((1 / curr_tau[j]) * diags(np.ones(S)))

        block_list.append((1 / 100) * diags(np.ones(3)))  # factor prior

        blocks = [[block_list[i] if i == j else None
                   for j in range(K + 1)]
                  for i in range(K + 1)]

        curr_prec = bmat(blocks, format="csr")

        XtOmega = X.T.multiply(omega)
        post_prec = XtOmega @ X + curr_prec
        post_prec = post_prec + 1e-8 * diags(np.ones(theta_dim))

        factor = cholesky(post_prec)
        mu = factor.solve_A(X.T @ kappa)
        curr_theta = mu + factor.solve_A(np.random.randn(theta_dim))

        # ---- update tau
        for j in range(K):
            sl = slice(j*S, (j+1)*S)
            beta = curr_theta[sl]
            quad = beta @ (prec @ beta) if j % 2 == 0 else beta @ beta
            curr_tau[j] = 1 / np.random.gamma(
                a_tau + S/2,
                1 / (b_tau + quad/2)
            )

        if it >= burn and (it - burn) % thin == 0:
            all_theta[:, save_idx] = curr_theta
            all_tau[:, save_idx]   = curr_tau
            save_idx += 1
            if save_idx == tot_save:
                break

    np.savez_compressed(save_path, all_theta=all_theta, all_tau=all_tau)

# ================================================================
# Run p01
# ================================================================
run_bym_factor(
    "p01",
    loc_mask=(y[:, :-1] == 0),
    kappa_transform=lambda ny: ny - 0.5,
    save_path=r"D:\77\Research\temp\snow\bym_factor_01_noiso.npz"
)

# ================================================================
# Run p10
# ================================================================
run_bym_factor(
    "p10",
    loc_mask=(y[:, :-1] == 1),
    kappa_transform=lambda ny: (1 - ny) - 0.5,
    save_path=r"D:\77\Research\temp\snow\bym_factor_10_noiso.npz"
)


Spatial locations used (non-isolated): 1601


MCMC p01:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_29976\3879063616.py:207: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(post_prec)
MCMC p10: 100%|█████████▉| 5995/6000 [3:08:21<00:09,  1.89s/it]  
